# Evaluation comparative Qwen3 medical SFT

Ce notebook telecharge `Maphe/medical-sft-5k`, isole le split de test, puis compare les generations de `unsloth/Qwen3-1.7B-unsloth-bnb-4bit` avec la version fine-tunee presente dans `./sft_output`.

Les reponses en texte libre sont evaluees avec METEOR. Les QCM sont evalues en verifiant uniquement si la premiere lettre de la reponse correspond a la reference (`OK` / `NOK`).

Le modele de base et le modele fine-tune sont charges l'un apres l'autre afin de limiter la VRAM.

Le deroulement suit une logique simple: verifier l'environnement, definir les parametres, preparer le dataset, lancer l'evaluation sur le modele de base, refaire exactement la meme evaluation sur le modele fine-tune, puis comparer les sorties de maniere quantitative et qualitative.

Le notebook est pense pour etre relance facilement sur Colab ou dans la `.venv` locale du projet. Les sections markdown ci-dessous precisent a chaque etape quelles variables sont critiques et comment interpreter les tableaux affiches.

## 1. Verification de l'environnement

Cette section confirme que le notebook tourne dans le bon environnement Python et que les dependances lourdes sont disponibles. Elle affiche aussi le GPU, CUDA et le support bf16, car ce sont les premiers points a verifier quand l'evaluation est lente ou plante au chargement du modele.

La cellule cherche d'abord automatiquement la racine du depot en remontant jusqu'au `pyproject.toml`. Cela evite les erreurs de chemin si le notebook est lance depuis `notebooks/`, depuis la racine du projet, ou depuis un environnement distant comme Colab.

Elle affiche ensuite les versions des bibliotheques critiques (`torch`, `transformers`, `unsloth`, `datasets`, etc.). Si une dependance apparait comme `MISSING`, il vaut mieux corriger l'environnement avant de continuer, car la suite du notebook depend directement de ces paquets.

Enfin, le rappel sur `.venv` sert de garde-fou: si le kernel n'utilise pas l'environnement uv du projet, vous pouvez obtenir des comportements incoherents entre le notebook et la ligne de commande.

In [1]:
import importlib.metadata as md
import os
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

print(f"Python executable : {sys.executable}")
print(f"Repo root         : {repo_root}")
print(f"Notebook cwd      : {Path.cwd()}")
print(f"HF endpoint       : {os.environ.get('HF_ENDPOINT', 'https://huggingface.co')}")

expected_packages = [
    "torch", "transformers", "unsloth", "unsloth_zoo",
    "bitsandbytes", "accelerate", "peft", "datasets", "pandas", "nltk",
]
for package in expected_packages:
    try:
        print(f"{package:16s}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package:16s}: MISSING")

if ".venv" not in sys.executable:
    print("\nAttention: le kernel ne semble pas utiliser la .venv uv du projet.")
    print("Demarrage recommande: uv run --with jupyter jupyter lab --port=8000 --no-browser --ip=0.0.0.0")

Python executable : /storage/user/zmxw1768/fine_tuning_oc/.venv/bin/python
Repo root         : /storage/user/zmxw1768/fine_tuning_oc
Notebook cwd      : /storage/user/zmxw1768/fine_tuning_oc/notebooks
HF endpoint       : https://huggingface.co
torch           : 2.10.0
transformers    : 4.57.6
unsloth         : 2026.5.2
unsloth_zoo     : 2026.5.1
bitsandbytes    : 0.49.2
accelerate      : 1.13.0
peft            : 0.19.1
datasets        : 4.3.0
pandas          : 3.0.2
nltk            : MISSING


## 2. Imports et configuration

On fixe ici les chemins, le modele de base, le dossier du fine-tune et les parametres d'evaluation. Les deux valeurs les plus sensibles pour le temps de calcul sont `EVAL_ROWS` et `MAX_NEW_TOKENS`; plus elles sont elevees, plus l'evaluation sera longue.

Cette cellule centralise aussi les variables d'environnement utiles a Unsloth. Le nettoyage prealable de `sys.modules` est important dans un notebook: apres un import partiel rate, Python peut conserver des modules incomplets en memoire et provoquer des erreurs difficiles a diagnostiquer au second essai.

Les constantes definies ici pilotent tout le reste du notebook: choix du dataset, modele de base, dossier LoRA, longueur maximale des sequences, taille de batch, nombre de lignes evaluees et emplacement des fichiers de sortie. Modifier cette cellule suffit generalement pour adapter l'evaluation a une autre machine ou a un autre checkpoint.

Le test CUDA a la fin force un echec explicite si aucun GPU n'est disponible. C'est preferable a une chute plus tardive pendant le chargement du modele, car la cause du probleme reste alors evidente.

In [2]:
from __future__ import annotations

from collections import Counter
import gc
import json
import random
import re
from time import perf_counter
from typing import Any

os.environ.setdefault("HF_ENDPOINT", "https://huggingface.co")
os.environ.setdefault("UNSLOTH_STABLE_DOWNLOADS", "1")
os.environ.setdefault("UNSLOTH_COMPILE_DISABLE", "1")  # Plus rapide pour les smoke tests d'evaluation.

# Si un import Unsloth a echoue plus tot dans ce kernel, Python peut garder des
# modules partiellement charges. On les purge avant le vrai import.
for module_name in list(sys.modules):
    if (
        module_name == "unsloth"
        or module_name.startswith("unsloth.")
        or module_name == "unsloth_zoo"
        or module_name.startswith("unsloth_zoo.")
    ):
        del sys.modules[module_name]

# Unsloth doit etre importe avant transformers / peft pour appliquer ses patchs.
from unsloth import FastModel, is_bfloat16_supported

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, load_dataset

DATASET_ID = "Maphe/medical-sft-5k"
BASE_MODEL_NAME = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit"
SFT_OUTPUT_CANDIDATES = [
    Path("./sft_output"),
    repo_root / "notebooks" / "sft_output",
    repo_root / "sft_output",
]
SFT_OUTPUT_DIR = next((path for path in SFT_OUTPUT_CANDIDATES if path.exists()), SFT_OUTPUT_CANDIDATES[0])

MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = True
DEVICE_MAP = "auto"  # "auto" est plus direct sur RTX 3090; utiliser "sequential" si VRAM limitee.
SEED = 42

# Mettre None pour evaluer tout le split de test. Garder une valeur basse pour un smoke test rapide.
EVAL_ROWS: int | None = 500
TEXT_MAX_NEW_TOKENS = 128
QCM_MAX_NEW_TOKENS = 8
MAX_NEW_TOKENS = TEXT_MAX_NEW_TOKENS
MAX_INPUT_TOKENS = MAX_SEQ_LENGTH - TEXT_MAX_NEW_TOKENS
EVAL_BATCH_SIZE = 8
PROGRESS_EVERY = 10
SAVE_GENERATIONS = True
RESULTS_DIR = SFT_OUTPUT_DIR.parent / "eval_results"

SYSTEM_PROMPT = (
    "Tu es un assistant medical expert. "
    "Reponds de maniere claire, factuelle et structuree. "
    "Si la question est en anglais, reponds en anglais."
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU CUDA detecte. Dans Colab: Runtime > Change runtime type > GPU.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")
print(f"bf16 supporte: {is_bfloat16_supported()}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: NVIDIA GeForce RTX 3090
CUDA: 12.8
bf16 supporte: True


## 3. Helpers

Ces fonctions preparent les exemples, construisent les prompts, generent les reponses et calculent les metriques. La logique importante est la separation entre `texte_libre` et `qcm`: les textes libres utilisent maintenant METEOR, la similarite cosinus et la distance euclidienne, tandis que les QCM sont juges uniquement sur la premiere lettre produite.

On peut lire cette section comme le coeur fonctionnel du notebook. Les premieres fonctions nettoient le texte, verifient le schema du dataset et choisissent le bon checkpoint LoRA. Viennent ensuite les utilitaires de classification des exemples pour distinguer automatiquement les questions ouvertes des QCM.

La partie generation encapsule la construction du prompt chat, la tokenisation, l'appel a `model.generate` et le post-traitement des reponses. Le batching est gere separement pour amortir le cout GPU tout en conservant des parametres de generation differents entre `qcm` et `texte_libre`.

La fin de la cellule regroupe toute la logique de scoring. Pour le texte libre, METEOR mesure le recouvrement lexical, la similarite cosinus mesure l'alignement directionnel entre vecteurs de frequences de tokens, et la distance euclidienne mesure l'ecart absolu entre ces memes vecteurs. Pour les QCM, le critere reste volontairement strict et simple: seule la premiere lettre de la reponse compte.

In [3]:
def clean_text(value: Any) -> str:
    # Normalise les espaces pour rendre les comparaisons plus stables.
    text = "" if value is None else str(value)
    return " ".join(text.strip().split())


def require_columns(dataset: Dataset, columns: set[str], dataset_id: str) -> None:
    missing = columns - set(dataset.column_names)
    if missing:
        raise ValueError(
            f"Dataset {dataset_id} invalide. Colonnes manquantes: {sorted(missing)}. "
            f"Colonnes presentes: {dataset.column_names}"
        )


def latest_checkpoint(output_dir: Path) -> Path:
    # Accepte soit un dossier final directement exploitable, soit un dossier contenant plusieurs checkpoints.
    if not output_dir.exists():
        raise FileNotFoundError(f"Dossier introuvable: {output_dir.resolve()}")

    if (output_dir / "adapter_model.safetensors").exists():
        return output_dir

    checkpoints = []
    for path in output_dir.glob("checkpoint-*"):
        if path.is_dir() and (path / "adapter_model.safetensors").exists():
            match = re.search(r"checkpoint-(\d+)$", path.name)
            step = int(match.group(1)) if match else -1
            checkpoints.append((step, path))

    if not checkpoints:
        raise FileNotFoundError(
            f"Aucun checkpoint LoRA trouve dans {output_dir.resolve()} "
            "(attendu: adapter_model.safetensors)."
        )
    return max(checkpoints, key=lambda item: item[0])[1]


def select_test_split(raw_dataset: DatasetDict | Dataset) -> Dataset:
    # Priorise un vrai split d'evaluation, sinon cree un split test deterministe depuis train.
    if isinstance(raw_dataset, DatasetDict):
        preferred_splits = ["test", "validation", "eval"]
        for split_name in preferred_splits:
            if split_name in raw_dataset:
                print(f"Split utilise: {split_name}")
                return raw_dataset[split_name]

        if "train" not in raw_dataset:
            raise ValueError(f"Aucun split test/validation/train trouve. Splits: {list(raw_dataset)}")

        print("Aucun split test detecte: creation d'un split test deterministe depuis train (10%).")
        split = raw_dataset["train"].train_test_split(test_size=0.1, seed=SEED, shuffle=True)
        return split["test"]

    print("Dataset sans splits nommes: creation d'un split test deterministe (10%).")
    return raw_dataset.train_test_split(test_size=0.1, seed=SEED, shuffle=True)["test"]


CHOICE_OPTION_RE = re.compile(r"(?:^|[\n\r\s])([A-H])\s*[\).:-]\s*\S", re.IGNORECASE)
FIRST_LETTER_RE = re.compile(r"^\s*([A-H])\b", re.IGNORECASE)
WORD_RE = re.compile(r"\w+", re.UNICODE)


def extract_choice_options(text: str) -> set[str]:
    # Repere les choix du type 'A)', 'B.' ou 'C -' dans l'enonce.
    return {match.group(1).upper() for match in CHOICE_OPTION_RE.finditer(clean_text(text))}


def first_answer_letter(text: str) -> str | None:
    # Utilise uniquement la premiere lettre pour evaluer les QCM de maniere robuste.
    match = FIRST_LETTER_RE.match(clean_text(text))
    if not match:
        return None
    return match.group(1).upper()


def is_qcm_example(instruction: str, reference: str) -> bool:
    options = extract_choice_options(instruction)
    reference_letter = first_answer_letter(reference)
    return len(options) >= 2 or (reference_letter is not None and reference_letter in options)


def classify_question_type(instruction: str, reference: str) -> str:
    return "qcm" if is_qcm_example(instruction, reference) else "texte_libre"


def prepare_eval_rows(test_dataset: Dataset) -> pd.DataFrame:
    require_columns(test_dataset, {"instruction", "response"}, DATASET_ID)
    eval_dataset = test_dataset.shuffle(seed=SEED)
    if EVAL_ROWS is not None:
        eval_dataset = eval_dataset.select(range(min(EVAL_ROWS, len(eval_dataset))))

    # On convertit le dataset Hugging Face en lignes simples pour faciliter l'analyse pandas ensuite.
    rows = []
    for idx, example in enumerate(eval_dataset):
        rows.append(
            {
                "example_id": idx,
                "instruction": clean_text(example["instruction"]),
                "reference": clean_text(example["response"]),
            }
        )
    for row in rows:
        row["task_type"] = classify_question_type(row["instruction"], row["reference"])
        row["reference_qcm_letter"] = first_answer_letter(row["reference"]) if row["task_type"] == "qcm" else None
    return pd.DataFrame(rows)


def build_prompt(tokenizer: Any, instruction: str, task_type: str | None = None) -> str:
    user_content = clean_text(instruction)
    if task_type == "qcm":
        # On contraint explicitement la sortie a une lettre pour limiter les generations bavardes.
        user_content = f"{user_content}\n\nReponds uniquement par la lettre de la bonne reponse."
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    kwargs = {"tokenize": False, "add_generation_prompt": True}
    try:
        # Certains tokenizers Qwen3 exposent enable_thinking, d'autres non.
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kwargs)


def apply_prompt(tokenizer: Any, instruction: str, task_type: str | None = None) -> dict[str, torch.Tensor]:
    # Tronque cote entree pour garder de la place a la generation dans la fenetre de contexte.
    prompt = build_prompt(tokenizer, instruction, task_type)
    return tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )


def strip_thinking_markers(text: str) -> str:
    # Nettoie les traces de raisonnement interne si le modele en produit malgre la configuration.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return text.replace("<think>\n\n</think>", "").strip()


def generate_answer(
    model: Any,
    tokenizer: Any,
    instruction: str,
    task_type: str | None = None,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> str:
    # Helper unitaire pratique pour debugger un exemple isole hors boucle batch.
    FastModel.for_inference(model)
    inputs = {key: value.to(model.device) for key, value in apply_prompt(tokenizer, instruction, task_type).items()}
    prompt_length = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.12,
            no_repeat_ngram_size=5,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True).strip()
    return strip_thinking_markers(reply)


def generate_answers_batch(
    model: Any,
    tokenizer: Any,
    instructions: list[str],
    task_types: list[str],
    max_new_tokens: int,
) -> list[str]:
    if not instructions:
        return []

    # Les prompts sont prepares en amont pour reutiliser la meme logique que le mode unitaire.
    prompts = [
        build_prompt(tokenizer, instruction, task_type)
        for instruction, task_type in zip(instructions, task_types)
    ]
    previous_padding_side = getattr(tokenizer, "padding_side", "right")
    # Padding a gauche pour aligner la fin des prompts dans le batch causal.
    tokenizer.padding_side = "left"
    try:
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            add_special_tokens=False,
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        )
    finally:
        tokenizer.padding_side = previous_padding_side

    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    # Tous les prompts sont paddes a la meme longueur, on coupe donc la sortie a partir de cette borne.
    prompt_length = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.12,
            no_repeat_ngram_size=5,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        )
    replies = tokenizer.batch_decode(outputs[:, prompt_length:], skip_special_tokens=True)
    return [strip_thinking_markers(reply.strip()) for reply in replies]


def generate_predictions(model: Any, tokenizer: Any, frame: pd.DataFrame, label: str) -> tuple[list[str], float]:
    FastModel.for_inference(model)
    rows = list(frame.itertuples(index=False))
    total = len(frame)
    predictions: list[str | None] = [None] * total
    # Les QCM et les textes libres n'ont pas le meme budget de generation.
    task_max_tokens = {"qcm": QCM_MAX_NEW_TOKENS, "texte_libre": TEXT_MAX_NEW_TOKENS}
    processed = 0
    t0 = perf_counter()
    for task_type, max_new_tokens in task_max_tokens.items():
        # On conserve l'ordre d'origine des exemples en memorisant leur position dans le DataFrame.
        positions = [idx for idx, row in enumerate(rows) if row.task_type == task_type]
        for start in range(0, len(positions), EVAL_BATCH_SIZE):
            batch_t0 = perf_counter()
            batch_positions = positions[start : start + EVAL_BATCH_SIZE]
            batch_rows = [rows[position] for position in batch_positions]
            batch_predictions = generate_answers_batch(
                model,
                tokenizer,
                [row.instruction for row in batch_rows],
                [row.task_type for row in batch_rows],
                max_new_tokens=max_new_tokens,
            )
            for position, prediction in zip(batch_positions, batch_predictions):
                predictions[position] = prediction

            processed += len(batch_positions)
            if PROGRESS_EVERY and (processed <= EVAL_BATCH_SIZE or processed % PROGRESS_EVERY == 0 or processed == total):
                elapsed = perf_counter() - t0
                per_item = elapsed / processed
                remaining = per_item * (total - processed)
                print(
                    f"{label}: {processed}/{total} - batch {task_type} de {len(batch_positions)} en "
                    f"{perf_counter() - batch_t0:.1f}s, {per_item:.1f}s/exemple, reste ~{remaining:.0f}s",
                    flush=True,
                )

    return [prediction or "" for prediction in predictions], perf_counter() - t0


def unload_model(model: Any | None = None, tokenizer: Any | None = None) -> None:
    # Liberation explicite pour reduire le risque d'OOM entre le modele de base et le fine-tune.
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def tokenize_for_meteor(text: str) -> list[str]:
    # Tokenisation simple et deterministe suffisante pour le calcul METEOR ici.
    return WORD_RE.findall(clean_text(text).lower())


def token_frequency_vector(text: str, vocabulary: list[str]) -> np.ndarray:
    # Vecteur de frequences normalise pour comparer deux textes sans dependance externe.
    token_counts: Counter[str] = Counter(tokenize_for_meteor(text))
    total = float(sum(token_counts.values()))
    if total == 0.0:
        return np.zeros(len(vocabulary), dtype=float)
    return np.array([token_counts[token] / total for token in vocabulary], dtype=float)


def text_vector_metrics(prediction: str, reference: str) -> tuple[float, float]:
    vocabulary = sorted(set(tokenize_for_meteor(prediction)) | set(tokenize_for_meteor(reference)))
    if not vocabulary:
        return 1.0, 0.0

    prediction_vector = token_frequency_vector(prediction, vocabulary)
    reference_vector = token_frequency_vector(reference, vocabulary)
    prediction_norm = float(np.linalg.norm(prediction_vector))
    reference_norm = float(np.linalg.norm(reference_vector))

    if prediction_norm == 0.0 and reference_norm == 0.0:
        cosine_similarity = 1.0
    elif prediction_norm == 0.0 or reference_norm == 0.0:
        cosine_similarity = 0.0
    else:
        cosine_similarity = float(
            np.dot(prediction_vector, reference_vector) / (prediction_norm * reference_norm)
        )

    euclidean_distance = float(np.linalg.norm(prediction_vector - reference_vector))
    return cosine_similarity, euclidean_distance


def meteor_exact(prediction: str, reference: str) -> float:
    # Fallback local inspire de METEOR si NLTK n'est pas disponible dans l'environnement.
    pred_tokens = tokenize_for_meteor(prediction)
    ref_tokens = tokenize_for_meteor(reference)
    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0

    used_ref = set()
    matches = []
    for pred_idx, pred_token in enumerate(pred_tokens):
        for ref_idx, ref_token in enumerate(ref_tokens):
            if ref_idx not in used_ref and pred_token == ref_token:
                used_ref.add(ref_idx)
                matches.append((pred_idx, ref_idx))
                break

    match_count = len(matches)
    if match_count == 0:
        return 0.0

    # METEOR privilegie le rappel via une moyenne ponderee precision/rappel.
    precision = match_count / len(pred_tokens)
    recall = match_count / len(ref_tokens)
    f_mean = (10 * precision * recall) / (recall + 9 * precision)

    # Plus les correspondances sont fragmentees, plus la penalite augmente.
    chunks = 1
    for (_, prev_ref_idx), (_, ref_idx) in zip(matches, matches[1:]):
        if ref_idx != prev_ref_idx + 1:
            chunks += 1
    penalty = 0.5 * (chunks / match_count) ** 3
    return float(f_mean * (1 - penalty))


def meteor_score_text(prediction: str, reference: str) -> float:
    try:
        from nltk.translate.meteor_score import single_meteor_score

        return float(single_meteor_score(tokenize_for_meteor(reference), tokenize_for_meteor(prediction)))
    except Exception:
        # On degrade proprement vers une implementation locale plutot que d'arreter l'evaluation.
        return meteor_exact(prediction, reference)


def qcm_ok_nok(prediction: str, reference: str) -> tuple[str, str | None]:
    # Le scoring QCM ignore tout sauf la premiere lettre detectee.
    prediction_letter = first_answer_letter(prediction)
    reference_letter = first_answer_letter(reference)
    if prediction_letter is None or reference_letter is None:
        return "NOK", prediction_letter
    return ("OK" if prediction_letter == reference_letter else "NOK"), prediction_letter


def add_metrics(frame: pd.DataFrame, prediction_column: str, prefix: str) -> pd.DataFrame:
    meteor_scores = []
    cosine_scores = []
    euclidean_distances = []
    qcm_scores = []
    qcm_letters = []
    qcm_statuses = []

    # Chaque ligne recoit soit des scores texte, soit un score QCM, selon sa nature.
    for row in frame.itertuples(index=False):
        prediction = getattr(row, prediction_column)
        reference = row.reference
        if row.task_type == "qcm":
            status, prediction_letter = qcm_ok_nok(prediction, reference)
            meteor_scores.append(np.nan)
            cosine_scores.append(np.nan)
            euclidean_distances.append(np.nan)
            qcm_scores.append(1.0 if status == "OK" else 0.0)
            qcm_letters.append(prediction_letter)
            qcm_statuses.append(status)
        else:
            cosine_similarity, euclidean_distance = text_vector_metrics(prediction, reference)
            meteor_scores.append(meteor_score_text(prediction, reference))
            cosine_scores.append(cosine_similarity)
            euclidean_distances.append(euclidean_distance)
            qcm_scores.append(np.nan)
            qcm_letters.append(None)
            qcm_statuses.append(None)

    frame[f"{prefix}_meteor"] = meteor_scores
    frame[f"{prefix}_cosine_similarity"] = cosine_scores
    frame[f"{prefix}_euclidean_distance"] = euclidean_distances
    frame[f"{prefix}_qcm_score"] = qcm_scores
    frame[f"{prefix}_qcm_letter"] = qcm_letters
    frame[f"{prefix}_qcm_ok_nok"] = qcm_statuses
    frame[f"{prefix}_chars"] = frame[prediction_column].str.len()
    return frame


def pairwise_rank(
    base_value: float,
    finetuned_value: float,
    *,
    higher_is_better: bool,
) -> tuple[float, float]:
    if pd.isna(base_value) or pd.isna(finetuned_value):
        return np.nan, np.nan
    if np.isclose(base_value, finetuned_value, equal_nan=False):
        return 1.0, 1.0

    base_is_better = base_value > finetuned_value if higher_is_better else base_value < finetuned_value
    return (1.0, 2.0) if base_is_better else (2.0, 1.0)


def add_pairwise_rank_columns(
    frame: pd.DataFrame,
    metric_name: str,
    *,
    higher_is_better: bool,
) -> pd.DataFrame:
    base_ranks = []
    finetuned_ranks = []
    base_column = f"base_{metric_name}"
    finetuned_column = f"finetuned_{metric_name}"

    for base_value, finetuned_value in zip(frame[base_column], frame[finetuned_column]):
        base_rank, finetuned_rank = pairwise_rank(
            base_value,
            finetuned_value,
            higher_is_better=higher_is_better,
        )
        base_ranks.append(base_rank)
        finetuned_ranks.append(finetuned_rank)

    frame[f"base_rank_{metric_name}"] = base_ranks
    frame[f"finetuned_rank_{metric_name}"] = finetuned_ranks
    return frame

## 4. Telechargement du dataset et preparation du test

On charge le dataset Hugging Face, puis on choisit un split de test. Si aucun split de test n'existe, un split deterministe est cree depuis `train`. La table `eval_frame` contient ensuite les instructions, les references et le type de question detecte.

La cellule affiche d'abord la structure brute du dataset pour verifier les splits exposes par Hugging Face. Cela permet de voir immediatement si l'on travaille sur un `DatasetDict` classique ou sur une structure plus simple necessitant une decoupe manuelle.

`prepare_eval_rows` normalise ensuite les colonnes utiles, melange les exemples avec une graine fixe et applique eventuellement la limite `EVAL_ROWS`. Le resultat final est un `DataFrame` pandas plus pratique pour l'evaluation, l'analyse et l'export que le format natif `datasets`.

Les compteurs affiches a la fin servent de verification rapide: ils confirment le nombre de lignes effectivement evaluees et la repartition entre questions ouvertes et QCM. Si cette repartition semble aberrante, il faut d'abord revoir la logique de detection avant d'interpreter les scores.

In [4]:
raw_dataset = load_dataset(DATASET_ID)
print(raw_dataset)

test_dataset = select_test_split(raw_dataset)
eval_frame = prepare_eval_rows(test_dataset)

print(f"Lignes dans le split test source: {len(test_dataset)}")
print(f"Lignes evaluees: {len(eval_frame)}")
print("Types de questions:")
print(eval_frame["task_type"].value_counts(dropna=False))
eval_frame.head(3)

DatasetDict({
    train: Dataset({
        features: ['dataset', 'source_family', 'source_repo_id', 'source_config', 'split', 'source_id', 'language', 'task_type', 'topic', 'instruction', 'response', 'answer_key', 'answer_index'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['dataset', 'source_family', 'source_repo_id', 'source_config', 'split', 'source_id', 'language', 'task_type', 'topic', 'instruction', 'response', 'answer_key', 'answer_index'],
        num_rows: 500
    })
    test: Dataset({
        features: ['dataset', 'source_family', 'source_repo_id', 'source_config', 'split', 'source_id', 'language', 'task_type', 'topic', 'instruction', 'response', 'answer_key', 'answer_index'],
        num_rows: 500
    })
})
Split utilise: test
Lignes dans le split test source: 500
Lignes evaluees: 500
Types de questions:
task_type
texte_libre    267
qcm            233
Name: count, dtype: int64


,example_id,instruction,reference,task_type,reference_qcm_letter
0,0,Réponds à la demande médicale suivante de mani...,L’association 5FU/acide folinique potentialise...,texte_libre,NaN
1,1,Réponds à la question médicale suivante en don...,C. Thrombo-phlébite du sinus latéral,qcm,C
2,2,Réponds à la demande médicale suivante de mani...,Origine de la contamination de Coralie T. : D’...,texte_libre,NaN


## 5. Evaluation du modele de base

Cette cellule charge le modele Qwen3 de reference, genere ses reponses sur les exemples d'evaluation, puis calcule les scores. Le modele est ensuite decharge de la VRAM avant de charger le fine-tune.

Le chargement passe par `FastModel.from_pretrained` avec quantification 4 bits, ce qui reduit fortement la consommation memoire par rapport a un chargement pleine precision. Si le tokenizer n'a pas de `pad_token`, on le force sur `eos_token` pour eviter les erreurs de padding pendant l'inference par batch.

`generate_predictions` parcourt ensuite tous les exemples, separe les QCM et les textes libres, puis applique le nombre de tokens adequat a chaque type de tache. Le temps total et le temps moyen par exemple sont affiches pour donner une estimation du cout de la boucle d'inference.

Les colonnes ajoutees a `eval_frame` a cette etape constituent la ligne de base de toute la comparaison. Il est donc utile de verifier rapidement quelques reponses avant de poursuivre, surtout si les generations paraissent vides, tronquees ou hors sujet.

In [5]:
base_model, base_tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=False,
    full_finetuning=False,
    dtype=None,
    device_map=DEVICE_MAP,
)

if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_predictions, elapsed = generate_predictions(base_model, base_tokenizer, eval_frame, "Base")

eval_frame["base_answer"] = base_predictions
eval_frame = add_metrics(eval_frame, "base_answer", "base")
print(f"Base termine en {elapsed:.1f}s ({elapsed / max(len(eval_frame), 1):.2f}s/exemple)")

unload_model(base_model, base_tokenizer)

==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 24.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/Qwen3-1.7B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Base: 8/500 - batch qcm de 8 en 1.0s, 0.1s/exemple, reste ~62s
Base: 40/500 - batch qcm de 8 en 0.7s, 0.1s/exemple, reste ~40s
Base: 80/500 - batch qcm de 8 en 0.7s, 0.1s/exemple, reste ~36s
Base: 120/500 - batch qcm de 8 en 0.7s, 0.1s/exemple, reste ~33s
Base: 160/500 - batch qcm de 8 en 0.6s, 0.1s/exemple, reste ~28s
Base: 200/500 - batch qcm de 8 en 0.7s, 0.1s/exemple, reste ~25s
Base: 500/500 - batch texte_libre de 3 en 7.7s, 

## 6. Evaluation du modele fine-tune depuis sft_output

On recupere le dernier checkpoint LoRA disponible dans `sft_output`, puis on applique exactement la meme boucle d'evaluation que pour le modele de base. Les scores sont donc comparables ligne par ligne.

La fonction `latest_checkpoint` choisit automatiquement le checkpoint numeriquement le plus avance contenant un `adapter_model.safetensors`. Cela evite de devoir modifier le notebook apres chaque session d'entrainement, tant que les sorties restent dans l'arborescence attendue.

L'evaluation qui suit reutilise exactement les memes helpers, les memes prompts et les memes metriques que pour le modele de base. Cette symetrie est essentielle: si l'on changeait aussi la preparation ou le scoring, la comparaison ne mesurerait plus uniquement l'effet du fine-tuning.

Comme pour la cellule precedente, les nouvelles colonnes ajoutees a `eval_frame` contiennent a la fois les reponses brutes et les metriques derivees. Le notebook dispose alors de toutes les informations necessaires pour comparer les deux modeles sans refaire d'inference.

In [6]:
finetuned_path = latest_checkpoint(SFT_OUTPUT_DIR)
print(f"Checkpoint fine-tune utilise: {finetuned_path.resolve()}")

finetuned_model, finetuned_tokenizer = FastModel.from_pretrained(
    model_name=str(finetuned_path),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=False,
    full_finetuning=False,
    dtype=None,
    device_map=DEVICE_MAP,
)

if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

finetuned_predictions, elapsed = generate_predictions(finetuned_model, finetuned_tokenizer, eval_frame, "Fine-tune")

eval_frame["finetuned_answer"] = finetuned_predictions
eval_frame = add_metrics(eval_frame, "finetuned_answer", "finetuned")
print(f"Fine-tune termine en {elapsed:.1f}s ({elapsed / max(len(eval_frame), 1):.2f}s/exemple)")

unload_model(finetuned_model, finetuned_tokenizer)

Checkpoint fine-tune utilise: /storage/user/zmxw1768/fine_tuning_oc/notebooks/sft_output/checkpoint-625
==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 24.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Fine-tune: 8/500 - batch qcm de 8 en 0.9s, 0.1s/exemple, reste ~56s
Fine-tune: 40/500 - batch qcm de 8 en 0.9s, 0.1s/exemple, reste ~49s
Fine-tune: 80/500 - batch qcm de 8 en 0.9s, 0.1s/exemple, reste ~45s
Fine-tune: 120/500 - batch qcm de 8 en 0.9s, 0.1s/exemple, reste ~42s
Fine-tune: 160/500 - batch qcm de 8 en 0.8s, 0.1s/exemple, reste ~37s
Fine-tune: 200/500 - batch qcm de 8 en 0.9s, 0.1s/exemple, reste ~32s
Fine-tune: 500/50

## 7. Comparaison quantitative

Cette partie resume les performances avec une vue agregee. Pour le texte libre, une hausse de `finetuned_meteor` ou de `finetuned_cosine_similarity` indique que les reponses se rapprochent davantage des references. A l'inverse, une baisse de `finetuned_euclidean_distance` indique que la reponse fine-tunee s'ecarte moins du texte cible. Pour les QCM, `qcm_score` vaut `1` quand la premiere lettre est correcte et `0` sinon.

La premiere cellule calcule des statistiques globales sur les principales colonnes numeriques: moyenne, mediane et ecart-type. Le delta moyen du fine-tune par rapport au modele de base aide a voir rapidement si l'adaptation apporte un gain global, une regression, ou un effet marginal selon le type de mesure considere.

Il faut toutefois lire ces moyennes avec prudence. Une hausse des caracteres generes n'est pas forcement un progres, et une moyenne peut masquer des comportements tres differents entre sous-groupes d'exemples. C'est pour cela que la cellule suivante compte explicitement les gains, regressions et ex aequo exemple par exemple.

### METEOR

METEOR compare le recouvrement lexical entre la prediction et la reference. Plus le score est eleve, plus les mots importants et leur ordre restent proches de la reponse attendue. C'est une bonne mesure de fidelite textuelle, mais elle reste sensible aux reformulations.

### Similarite cosinus

La similarite cosinus est calculee ici sur des vecteurs de frequences de tokens normalises. Plus elle est proche de `1`, plus la prediction et la reference pointent dans la meme direction lexicale, meme si leur longueur exacte differe.

### Distance euclidienne

La distance euclidienne est calculee sur les memes vecteurs de frequences de tokens. Plus elle est faible, plus la prediction reste proche de la reference en termes de repartition globale des mots. C'est donc la seule metrique texte de cette section pour laquelle une baisse est une amelioration.

### Rang par exemple

Pour chaque exemple, le notebook attribue ensuite un rang au modele `base` et au modele `finetuned`. Sur les textes libres, ce rang global agrege les comparaisons par paire sur METEOR, similarite cosinus et distance euclidienne. Sur les QCM, le rang repose uniquement sur la justesse de la premiere lettre.

In [7]:
metric_columns = [
    "base_meteor", "finetuned_meteor",
    "base_cosine_similarity", "finetuned_cosine_similarity",
    "base_euclidean_distance", "finetuned_euclidean_distance",
    "base_qcm_score", "finetuned_qcm_score",
    "base_chars", "finetuned_chars",
]
summary = eval_frame[metric_columns].agg(["mean", "median", "std"]).T
summary["delta_mean_vs_base"] = np.nan
for metric_name, higher_is_better in [
    ("meteor", True),
    ("cosine_similarity", True),
    ("euclidean_distance", False),
    ("qcm_score", True),
    ("chars", True),
]:
    base_metric = f"base_{metric_name}"
    finetuned_metric = f"finetuned_{metric_name}"
    raw_delta = summary.loc[finetuned_metric, "mean"] - summary.loc[base_metric, "mean"]
    summary.loc[finetuned_metric, "delta_mean_vs_base"] = raw_delta if higher_is_better else -raw_delta

display(eval_frame["task_type"].value_counts().rename("n_examples"))
summary

task_type
texte_libre    267
qcm            233
Name: n_examples, dtype: int64

,mean,median,std,delta_mean_vs_base
base_meteor,0.136096,0.121019,0.091636,NaN
finetuned_meteor,0.165252,0.133685,0.152812,0.029156
base_cosine_similarity,0.376225,0.416675,0.197098,NaN
finetuned_cosine_similarity,0.385943,0.391652,0.250605,0.009719
base_euclidean_distance,0.227833,0.179118,0.161768,NaN
finetuned_euclidean_distance,0.223133,0.175189,0.166033,0.004701
base_qcm_score,0.051502,0.000000,0.221495,NaN
finetuned_qcm_score,0.437768,0.000000,0.497180,0.386266
base_chars,283.946000,389.000000,249.096723,NaN
finetuned_chars,286.730000,315.500000,263.526518,2.784000


Le tableau precedent donne les moyennes, medianes et ecarts-types. La cellule suivante complete cette lecture avec des comptes directs: combien d'exemples sont meilleurs avec le fine-tune, combien restent meilleurs avec le modele de base, et combien sont ex aequo.

Cette lecture par comptes est souvent plus parlante qu'une moyenne seule. Par exemple, un gain moyen legerement positif peut en realite venir d'un petit nombre de gros progres, alors que la majorite des exemples regressent. Inversement, une moyenne stable peut cacher une amelioration plus reguliere mais de faible amplitude.

La variable `delta_primary` conserve une metrique simple a lire rapidement: METEOR pour le texte libre, exactitude de la lettre pour les QCM. Des deltas specifiques sont aussi calcules pour la similarite cosinus et la distance euclidienne. Pour cette derniere, le signe est inverse afin qu'un delta positif signifie toujours un avantage pour le fine-tune.

Les colonnes de rang traduisent ensuite ces comparaisons a l'echelle de chaque exemple. Un rang `1` indique le meilleur modele, un rang `2` le moins bon, et `1` des deux cotes en cas d'egalite. Sur les textes libres, le rang global est derive de la moyenne des rangs obtenus sur METEOR, la similarite cosinus et la distance euclidienne.

In [8]:
eval_frame["delta_meteor"] = eval_frame["finetuned_meteor"] - eval_frame["base_meteor"]
eval_frame["delta_cosine_similarity"] = eval_frame["finetuned_cosine_similarity"] - eval_frame["base_cosine_similarity"]
eval_frame["delta_euclidean_distance"] = eval_frame["base_euclidean_distance"] - eval_frame["finetuned_euclidean_distance"]
eval_frame["delta_qcm_score"] = eval_frame["finetuned_qcm_score"] - eval_frame["base_qcm_score"]

eval_frame["base_primary_score"] = np.where(
    eval_frame["task_type"].eq("qcm"),
    eval_frame["base_qcm_score"],
    eval_frame["base_meteor"],
)
eval_frame["finetuned_primary_score"] = np.where(
    eval_frame["task_type"].eq("qcm"),
    eval_frame["finetuned_qcm_score"],
    eval_frame["finetuned_meteor"],
)
eval_frame["delta_primary"] = eval_frame["finetuned_primary_score"] - eval_frame["base_primary_score"]
eval_frame["primary_metric"] = np.where(
    eval_frame["task_type"].eq("qcm"),
    "qcm_first_letter",
    "meteor",
)

for metric_name, higher_is_better in [
    ("meteor", True),
    ("cosine_similarity", True),
    ("euclidean_distance", False),
    ("qcm_score", True),
]:
    eval_frame = add_pairwise_rank_columns(
        eval_frame,
        metric_name,
        higher_is_better=higher_is_better,
    )


def compute_overall_ranks(row: pd.Series) -> pd.Series:
    if row["task_type"] == "qcm":
        base_rank_score = float(row["base_rank_qcm_score"])
        finetuned_rank_score = float(row["finetuned_rank_qcm_score"])
    else:
        base_rank_score = float(
            np.nanmean(
                [
                    row["base_rank_meteor"],
                    row["base_rank_cosine_similarity"],
                    row["base_rank_euclidean_distance"],
                ]
            )
        )
        finetuned_rank_score = float(
            np.nanmean(
                [
                    row["finetuned_rank_meteor"],
                    row["finetuned_rank_cosine_similarity"],
                    row["finetuned_rank_euclidean_distance"],
                ]
            )
        )

    if np.isclose(base_rank_score, finetuned_rank_score, equal_nan=False):
        base_rank = 1.0
        finetuned_rank = 1.0
    elif base_rank_score < finetuned_rank_score:
        base_rank = 1.0
        finetuned_rank = 2.0
    else:
        base_rank = 2.0
        finetuned_rank = 1.0

    return pd.Series(
        {
            "base_rank_score": base_rank_score,
            "finetuned_rank_score": finetuned_rank_score,
            "base_rank": base_rank,
            "finetuned_rank": finetuned_rank,
        }
    )


rank_frame = eval_frame.apply(compute_overall_ranks, axis=1)
eval_frame[rank_frame.columns] = rank_frame

def comparison_counts(delta: pd.Series, metric_name: str) -> dict[str, int]:
    values = delta.dropna()
    return {
        f"n_{metric_name}": int(values.shape[0]),
        f"finetuned_better_{metric_name}": int((values > 0).sum()),
        f"base_better_{metric_name}": int((values < 0).sum()),
        f"ties_{metric_name}": int((values == 0).sum()),
    }


wins = {
    **comparison_counts(eval_frame["delta_meteor"], "meteor_texte_libre"),
    **comparison_counts(eval_frame["delta_cosine_similarity"], "cosine_similarity_texte_libre"),
    **comparison_counts(eval_frame["delta_euclidean_distance"], "euclidean_distance_texte_libre"),
    **comparison_counts(eval_frame["delta_qcm_score"], "qcm_first_letter"),
    "base_qcm_ok": int((eval_frame["base_qcm_ok_nok"] == "OK").sum()),
    "finetuned_qcm_ok": int((eval_frame["finetuned_qcm_ok_nok"] == "OK").sum()),
    "base_rank_1": int((eval_frame["base_rank"] == 1).sum()),
    "finetuned_rank_1": int((eval_frame["finetuned_rank"] == 1).sum()),
    "rank_ties": int(((eval_frame["base_rank"] == 1) & (eval_frame["finetuned_rank"] == 1)).sum()),
}
pd.Series(wins, name="comparaison")

n_meteor_texte_libre                               267
finetuned_better_meteor_texte_libre                139
base_better_meteor_texte_libre                     115
ties_meteor_texte_libre                             13
n_cosine_similarity_texte_libre                    267
finetuned_better_cosine_similarity_texte_libre     113
base_better_cosine_similarity_texte_libre          141
ties_cosine_similarity_texte_libre                  13
n_euclidean_distance_texte_libre                   267
finetuned_better_euclidean_distance_texte_libre    159
base_better_euclidean_distance_texte_libre         108
ties_euclidean_distance_texte_libre                  0
n_qcm_first_letter                                 233
finetuned_better_qcm_first_letter                   93
base_better_qcm_first_letter                         3
ties_qcm_first_letter                              137
base_qcm_ok                                         12
finetuned_qcm_ok                                   102
base_rank_

## 8. Inspection qualitative

Les scores globaux ne suffisent pas pour comprendre le comportement du modele. Cette section affiche les meilleurs gains, les plus fortes regressions et quelques exemples aleatoires afin de verifier si les differences numeriques correspondent a des reponses medicalement plus utiles.

Les tableaux affiches ici sont essentiels pour sortir d'une lecture purement statistique. Un meilleur score peut parfois correspondre a une reponse plus proche textuellement de la reference mais moins utile medicalement, tandis qu'une baisse de score peut cacher une reformulation correcte mais differente sur le plan lexical.

Les meilleurs gains permettent d'identifier ce que le fine-tuning a vraiment appris. Les pires regressions, elles, mettent souvent en evidence des effets de bord comme la sur-specialisation, les reponses trop courtes, les consignes de style trop rigides ou des erreurs sur les QCM.

Les colonnes ajoutees dans cette section montrent aussi les trois metriques texte et le rang global par exemple. Cela permet de voir rapidement si un gain METEOR est confirme ou non par la similarite cosinus et la distance euclidienne.

L'echantillon aleatoire sert enfin de controle de realite: il aide a verifier que l'impression generale ne depend pas uniquement des cas extremes.

In [9]:
display_columns = [
    "example_id", "task_type", "primary_metric", "delta_primary",
    "base_rank", "finetuned_rank", "base_rank_score", "finetuned_rank_score",
    "instruction", "reference",
    "reference_qcm_letter", "base_answer", "finetuned_answer",
    "base_meteor", "finetuned_meteor",
    "base_cosine_similarity", "finetuned_cosine_similarity",
    "base_euclidean_distance", "finetuned_euclidean_distance",
    "base_qcm_letter", "finetuned_qcm_letter",
    "base_qcm_ok_nok", "finetuned_qcm_ok_nok",
]

print("Meilleurs gains fine-tune vs base")
display(eval_frame.sort_values("delta_primary", ascending=False)[display_columns].head(5))

print("Pires regressions fine-tune vs base")
display(eval_frame.sort_values("delta_primary", ascending=True)[display_columns].head(5))

print("Exemples aleatoires")
display(eval_frame.sample(min(5, len(eval_frame)), random_state=SEED)[display_columns])

Meilleurs gains fine-tune vs base


,example_id,task_type,primary_metric,delta_primary,base_rank,finetuned_rank,base_rank_score,finetuned_rank_score,instruction,reference,...,base_meteor,finetuned_meteor,base_cosine_similarity,finetuned_cosine_similarity,base_euclidean_distance,finetuned_euclidean_distance,base_qcm_letter,finetuned_qcm_letter,base_qcm_ok_nok,finetuned_qcm_ok_nok
484,484,qcm,qcm_first_letter,1.0,2.0,1.0,2.0,1.0,Réponds à la question médicale suivante en don...,B. C'est une bactérie toujours immobile,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B,NOK,OK
482,482,qcm,qcm_first_letter,1.0,2.0,1.0,2.0,1.0,Réponds à la question médicale suivante en don...,D. Le noyau et le cytosol,...,NaN,NaN,NaN,NaN,NaN,NaN,E,D,NOK,OK
443,443,qcm,qcm_first_letter,1.0,2.0,1.0,2.0,1.0,Réponds à la question médicale suivante en don...,D. Capacité d'adhérence aux biomatériaux,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D,NOK,OK
434,434,qcm,qcm_first_letter,1.0,2.0,1.0,2.0,1.0,Réponds à la question médicale suivante en don...,B. Le groupement amine oriente les substitutio...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B,NOK,OK
437,437,qcm,qcm_first_letter,1.0,2.0,1.0,2.0,1.0,Réponds à la question médicale suivante en don...,"D. Ulcérovégétante, correspondant à un carcino...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D,NOK,OK


Pires regressions fine-tune vs base


,example_id,task_type,primary_metric,delta_primary,base_rank,finetuned_rank,base_rank_score,finetuned_rank_score,instruction,reference,...,base_meteor,finetuned_meteor,base_cosine_similarity,finetuned_cosine_similarity,base_euclidean_distance,finetuned_euclidean_distance,base_qcm_letter,finetuned_qcm_letter,base_qcm_ok_nok,finetuned_qcm_ok_nok
111,111,qcm,qcm_first_letter,-1.000000,1.0,2.0,1.0,2.0,Réponds à la question médicale suivante en don...,B. Amoxicilline,...,NaN,NaN,NaN,NaN,NaN,NaN,B,C,OK,NOK
411,411,qcm,qcm_first_letter,-1.000000,1.0,2.0,1.0,2.0,Réponds à la question médicale suivante en don...,E. Quinidiniques,...,NaN,NaN,NaN,NaN,NaN,NaN,E,C,OK,NOK
404,404,qcm,qcm_first_letter,-1.000000,1.0,2.0,1.0,2.0,Réponds à la question médicale suivante en don...,B. 200 à 500 mg,...,NaN,NaN,NaN,NaN,NaN,NaN,B,C,OK,NOK
122,122,texte_libre,meteor,-0.260552,1.0,2.0,1.0,2.0,"Answer the following medical request clearly, ...",What causes Majeed syndrome? Majeed syndrome i...,...,0.388598,0.128046,0.668342,0.522760,0.130618,0.175189,NaN,NaN,NaN,NaN
2,2,texte_libre,meteor,-0.248671,1.0,2.0,1.0,2.0,Réponds à la demande médicale suivante de mani...,Origine de la contamination de Coralie T. : D’...,...,0.288372,0.039701,0.639176,0.053352,0.139750,0.199938,NaN,NaN,NaN,NaN


Exemples aleatoires


,example_id,task_type,primary_metric,delta_primary,base_rank,finetuned_rank,base_rank_score,finetuned_rank_score,instruction,reference,...,base_meteor,finetuned_meteor,base_cosine_similarity,finetuned_cosine_similarity,base_euclidean_distance,finetuned_euclidean_distance,base_qcm_letter,finetuned_qcm_letter,base_qcm_ok_nok,finetuned_qcm_ok_nok
361,361,texte_libre,meteor,-0.040546,1.0,2.0,1.000000,2.000000,Réponds à la demande médicale suivante de mani...,"Il existe une hyperleucocytose importante, ain...",...,0.061038,0.020492,0.289673,0.069577,0.185238,0.235234,NaN,NaN,NaN,NaN
73,73,texte_libre,meteor,0.050186,1.0,2.0,1.333333,1.666667,Réponds à la demande médicale suivante de mani...,Le diagnostic le plus probable est celui d'un ...,...,0.118443,0.168630,0.322737,0.314999,0.169055,0.170188,NaN,NaN,NaN,NaN
374,374,texte_libre,meteor,-0.049132,1.0,2.0,1.000000,2.000000,Réponds à la demande médicale suivante de mani...,Pneumonie à pneumocoque compliquée d’une ménin...,...,0.119555,0.070423,0.217407,0.215761,0.266797,0.276907,NaN,NaN,NaN,NaN
155,155,texte_libre,meteor,0.011250,1.0,2.0,1.333333,1.666667,Réponds à la demande médicale suivante de mani...,L’analyse est réalisée en immunofluorescence a...,...,0.125187,0.136437,0.514743,0.421586,0.146469,0.154589,NaN,NaN,NaN,NaN
104,104,qcm,qcm_first_letter,0.000000,1.0,1.0,1.000000,1.000000,Réponds à la question médicale suivante en don...,A. Ampicilline-gentamicine,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C,NOK,NOK


## 9. Sauvegarde des resultats

La derniere cellule sauvegarde les generations completes, le resume des metriques et les comptes de comparaison. Ces fichiers permettent de relire les sorties sans relancer les modeles, ce qui economise beaucoup de temps GPU.

Trois formats sont produits. Le CSV est pratique pour une lecture rapide dans pandas, Excel ou LibreOffice. Le JSONL conserve une structure ligne par ligne bien adaptee a des traitements automatiques. Le JSON de resume rassemble les metriques agregees et les compteurs de comparaison dans un format leger et facile a versionner.

Les colonnes exportees incluent maintenant les metriques texte supplementaires (`cosine_similarity`, `euclidean_distance`) ainsi que les rangs attribues a `base` et `finetuned` pour chaque exemple. Vous pouvez donc refaire l'analyse hors notebook sans recalculer les generations.

Le dossier de sortie est cree automatiquement si necessaire. Si vous relancez plusieurs experiences, il peut etre utile de renommer ce dossier ou les fichiers generes afin d'eviter d'ecraser une evaluation precedente.

In [10]:
if SAVE_GENERATIONS:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    csv_path = RESULTS_DIR / "qwen3_base_vs_sft_output.csv"
    jsonl_path = RESULTS_DIR / "qwen3_base_vs_sft_output.jsonl"
    summary_path = RESULTS_DIR / "qwen3_base_vs_sft_output_summary.json"

    eval_frame.to_csv(csv_path, index=False)
    eval_frame.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)

    payload = {
        "dataset_id": DATASET_ID,
        "base_model_name": BASE_MODEL_NAME,
        "finetuned_path": str(finetuned_path),
        "eval_rows": len(eval_frame),
        "metrics": json.loads(summary.to_json(orient="index")),
        "wins": wins,
    }
    summary_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False))

    print(f"CSV    : {csv_path.resolve()}")
    print(f"JSONL  : {jsonl_path.resolve()}")
    print(f"Resume : {summary_path.resolve()}")

CSV    : /storage/user/zmxw1768/fine_tuning_oc/notebooks/eval_results/qwen3_base_vs_sft_output.csv
JSONL  : /storage/user/zmxw1768/fine_tuning_oc/notebooks/eval_results/qwen3_base_vs_sft_output.jsonl
Resume : /storage/user/zmxw1768/fine_tuning_oc/notebooks/eval_results/qwen3_base_vs_sft_output_summary.json
